<a href="https://colab.research.google.com/github/yianli0213/programming-language/blob/main/%E3%80%8CHW4_PTT_GoogleSheet_RAG_kpop_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [5]:
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai gradio


In [6]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe
import gradio as gr

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [14]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/18fLKRONFTBpA-TXxrGjkB53qjtrA27IwgkMNlk07fps/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_kpop_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_kpop_INDEX = "https://www.ptt.cc/bbs/KoreaStar/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [8]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：rag
🔗 https://docs.google.com/spreadsheets/d/18fLKRONFTBpA-TXxrGjkB53qjtrA27IwgkMNlk07fps/edit?usp=sharing


In [10]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：ptt_kpop_posts


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [12]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_kpop_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [15]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_movie(pages=1, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/1: https://www.ptt.cc/bbs/KoreaStar/index.html
✅ 本次爬到 15 篇文章
📌 Google Sheet 原本有 0 筆
✅ 已寫入 Google Sheet：15 筆
🔍 從 Google Sheet 重新讀回：15 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [16]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：15


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1767200122.A.91D,[閒聊] 韓星板 2026 一月 置底閒聊文 I,https://www.ptt.cc/bbs/KoreaStar/M.1767200122....,01/01,yoche2000,100,Thu Jan 1 00:55:16 2026,2026-06-15 12:24:57,這可能是我的最後一篇閒聊文\n(我會處理完去年的案子再走)\n\n祝各位追星人好板友新年快樂...
1,M.1767246572.A.E4B,[情報] 2026年04-06月 來台活動/售票資訊,https://www.ptt.cc/bbs/KoreaStar/M.1767246572....,01/01,elvissu,12,Thu Jan 1 13:49:30 2026,2026-06-15 12:24:55,2026年 來台活動簡表\n(FM=粉絲見面會/FS=粉絲簽售/CON=演唱會/SC=SHO...
2,M.1751339434.A.2F9,[公告] 置底檢舉區 (2507-2512),https://www.ptt.cc/bbs/KoreaStar/M.1751339434....,07/01,yoche2000,75,Tue Jul 1 11:10:26 2025,2026-06-15 12:24:54,上半年那篇累積的案件已處理完畢，晚點會公告處分。\n先換篇。\n\n置底檢舉文：以下幾點公告...
3,M.1693884697.A.D5A,[公告] 韓星板板規 (板標投稿請站內板主),https://www.ptt.cc/bbs/KoreaStar/M.1693884697....,09/05,yoche2000,0,Tue Sep 5 11:31:31 2023,2026-06-15 12:24:52,※ 230904 根據\n#1awqfvQb\n納入該修正案及\n#1awqfvQb\n修正...
4,M.1781521942.A.EC3,[新聞] 中央集團副會長：JTBC申請企業重整,https://www.ptt.cc/bbs/KoreaStar/M.1781521942....,6/15,k721102,0,Mon Jun 15 19:12:18 2026,2026-06-15 12:24:50,中央集團副會長：JTBC申請企業重整不可避免\n\n韓聯社首爾6月15日電 針對JTBC電視...


In [17]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：15 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [18]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [20]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT kpopstar版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [23]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


請輸入問題：aespa
根據【PTT 資料】，關於 aespa 的相關資訊如下：

1.  **購票政策變更**：SM 娛樂針對 aespa 的韓國場演唱會開始實施「韓版優先購票」制度。韓版頁面的購票時間比國際版早約 4-5 天，這導致海外粉絲可能僅能購買剩餘的票券。
2.  **台灣演唱會資訊**：根據 2026 年來台活動簡表，aespa 預計於 2026 年 8 月 11 日（二）18:30 在「台北大巨蛋」舉行演唱會。

**參考來源：**
*   [閒聊] SM藝人（RV, aespa)開始韓版優先購票
    https://www.ptt.cc/bbs/KoreaStar/M.1781497705.A.95E.html
*   [情報] 2026年04-06月 來台活動/售票資訊
    https://www.ptt.cc/bbs/KoreaStar/M.1767246572.A.E4B.html


In [29]:
def analyze_star(star_name):
    if not star_name.strip():
        return "請輸入藝人名稱。", None

    # 篩選有提到這個藝人的文章
    df = read_sheet_df(ws_ptt, PTT_HEADER)
    mask = (
        df["title"].str.contains(star_name, case=False, na=False) |
        df["content"].str.contains(star_name, case=False, na=False)
    )
    filtered = df[mask].copy()

    if filtered.empty:
        return f"找不到任何提到「{star_name}」的文章。", None

    # 整理顯示用的 DataFrame
    display_df = filtered[["title","author","date","nrec","url"]].copy()
    display_df.columns = ["標題","作者","日期","推文數","網址"]

    # 組 prompt 給 Gemini 做分析
    articles_text = "\n\n---\n\n".join([
        f"標題：{row['title']}\n推文數：{row['nrec']}\n內容：{row['content'][:300]}"
        for _, row in filtered.iterrows()
    ])

    prompt = f"""
以下是 PTT KoreaStar 版上提到「{star_name}」的文章，請根據這些資料：
1. 摘要最近關於{star_name}的主要話題
2. 分析 PTT 網友對{star_name}的整體評價（正面/負面/中立）
3. 列出最值得關注的一篇文章標題與原因

請用繁體中文回答，條列清楚。

【PTT 文章資料】
{articles_text}

【分析結果】
""".strip()

    response = llm.generate_content(prompt)
    summary = f"📊 找到 {len(filtered)} 篇相關文章\n\n" + response.text

    return summary, display_df


with gr.Blocks(theme=gr.themes.Glass(), title="KoreaStar PTT 分析平台") as demo:
    gr.Markdown("# 🌟 KoreaStar PTT 情報站")
    gr.Markdown("爬取 PTT KoreaStar 版文章，用 AI 幫你分析韓星動態")

    with gr.Tab("🔍 明星分析"):
        gr.Markdown("## 輸入藝人名稱，AI 幫你分析 PTT 網友怎麼說")
        star_input = gr.Textbox(
            label="藝人名稱",
            placeholder="例如：BTS、BLACKPINK、IU、aespa...",
            lines=1
        )
        star_btn = gr.Button("開始分析", variant="primary")
        star_summary = gr.Textbox(label="AI 分析結果", lines=15, interactive=False)
        star_articles = gr.Dataframe(
            headers=["標題","作者","日期","推文數","網址"],
            label="相關文章列表",
            interactive=False,
            wrap=True
        )
        star_btn.click(
            fn=analyze_star,
            inputs=star_input,
            outputs=[star_summary, star_articles]
        )

    with gr.Tab("📡 PTT 爬蟲"):
        gr.Markdown("## 爬取最新 PTT 文章並更新資料庫")
        with gr.Row():
            crawler_pages = gr.Slider(minimum=1, maximum=10, value=2, step=1, label="爬取頁數")
            crawler_delay = gr.Slider(minimum=1.0, maximum=10.0, value=3.0, step=0.5, label="延遲（秒）")
        crawl_button = gr.Button("開始爬取並更新 RAG", variant="primary")
        crawl_status = gr.Textbox(label="狀態", lines=5, interactive=False)
        crawl_button.click(
            fn=run_crawler_and_update_rag,
            inputs=[crawler_pages, crawler_delay],
            outputs=crawl_status
        )

    with gr.Tab("📰 瀏覽文章"):
        gr.Markdown("## 查看所有已爬取的文章")
        refresh_btn = gr.Button("重新載入", variant="secondary")
        articles_df = gr.Dataframe(
            headers=["標題","作者","日期","推文數","網址"],
            interactive=False,
            wrap=True
        )
        refresh_btn.click(fn=get_articles_for_display, outputs=articles_df)
        demo.load(get_articles_for_display, None, articles_df)

    with gr.Tab("💬 RAG 問答"):
        gr.Markdown("## 直接問 AI，根據 PTT 文章回答你")
        rag_input = gr.Textbox(
            label="你的問題",
            lines=2,
            placeholder="例如：最近韓星有什麼新聞？哪個團體最多人討論？"
        )
        rag_btn = gr.Button("送出問題", variant="primary")
        rag_output = gr.Textbox(label="AI 回答", lines=15, interactive=False)
        rag_btn.click(fn=gradio_query_rag, inputs=rag_input, outputs=rag_output)

demo.launch(debug=True, share=True)

/tmp/ipykernel_3706/2432195872.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Glass(), title="KoreaStar PTT 分析平台") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://32283f5b9c81473388.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/KoreaStar/index3462.html
📄 正在讀取列表頁 1/1: https://www.ptt.cc/bbs/KoreaStar/index.html
✅ 本次爬到 35 篇文章


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 本次爬到 15 篇文章


...

✅ 本次爬到 35 篇文章


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

📄 正在讀取列表頁 1/1: https://www.ptt.cc/bbs/KoreaStar/index.html
✅ 本次爬到 15 篇文章


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://32283f5b9c81473388.gradio.live


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。
